# 手元の LLM に生成文を採点させる (LLM-as-a-judge)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_llm_analysis/llm_judge_local.ipynb)

NLP クックブック 23-02 ／ LLM の評価と分析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

要約や応答の品質を LLM に採点させたい。API を使わずに手元の小さなモデルで行い、その採点がどれくらい信用できるかも確かめたい。

In [1]:
import numpy as np
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル (T4 GPU で 1 件あたり 1 秒未満)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()
DIGITS = [tokenizer(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, 6)]

@torch.no_grad()
def next_token_probs(prompt, token_ids):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(llm.device)
    return llm(ids).logits[0, -1, token_ids].float().softmax(-1).cpu()

RUBRIC = ("あなたは要約の評価者です。「元の文章」と「要約」を読み、要約の忠実性を 1〜5 の整数で採点してください。\n"
          "5: 要約の内容はすべて元の文章に書かれている\n3: おおむね正しいが、元の文章にない情報や不正確な点が少しある\n1: 元の文章と矛盾する、または無関係である\n"
          "数字だけを答えてください。")

def score(source, summary):
    """1〜5 の各数字が次に出る確率から、期待値をスコアとする (生成させるより細かく、安定した値になる)"""
    probs = next_token_probs(f"{RUBRIC}\n\n# 元の文章\n{source}\n\n# 要約\n{summary}", DIGITS)
    return float((probs * torch.arange(1, 6)).sum())

source = ("市は 14 日、来年 4 月から市内の全小中学校で給食費を無償化すると発表した。対象は約 1 万 2 千人で、年間の費用は約 6 億円を見込む。"
          "財源には、ふるさと納税の寄付金と、公共施設の統廃合で生じる削減分を充てる。一方、市議会の一部からは「恒久的な財源の裏付けが弱い」との指摘が出ている。")
items = [  # (要約, 人間が付けた忠実性の評点)
    ("市は来年 4 月から小中学校の給食費を無償化する。費用は年間約 6 億円を見込む。", 5),
    ("市は給食費の無償化を発表した。市議会の一部からは財源を不安視する声がある。", 5),
    ("市は来年 4 月から給食費を無償化する。対象は約 1 万 2 千人である。", 5),
    ("市は給食費を無償化する。財源は国の補助金でまかなう。", 2),
    ("市は来年 4 月から給食費を無償化する。保護者の 9 割が賛成している。", 3),
    ("市は給食費の無償化を発表した。市議会は全会一致で賛成した。", 1),
    ("市は来年 4 月から給食費を値上げすると発表した。", 1),
    ("市は高校の授業料を無償化する。費用は年間約 60 億円である。", 1),
    ("県は来月から新しい図書館を開館する。", 1),
    ("市は給食費を無償化する。費用は約 6 億円で、寄付金などを充てる。", 5),
]
scores = [score(source, s) for s, _ in items]
for (s, human), llm_score in zip(items, scores):
    print(f"人 {human} / LLM {llm_score:.2f} | {s}")

人 5 / LLM 3.00 | 市は来年 4 月から小中学校の給食費を無償化する。費用は年間約 6 億円を見込む。
人 5 / LLM 3.00 | 市は給食費の無償化を発表した。市議会の一部からは財源を不安視する声がある。
人 5 / LLM 3.00 | 市は来年 4 月から給食費を無償化する。対象は約 1 万 2 千人である。
人 2 / LLM 3.00 | 市は給食費を無償化する。財源は国の補助金でまかなう。
人 3 / LLM 3.00 | 市は来年 4 月から給食費を無償化する。保護者の 9 割が賛成している。
人 1 / LLM 3.00 | 市は給食費の無償化を発表した。市議会は全会一致で賛成した。
人 1 / LLM 2.99 | 市は来年 4 月から給食費を値上げすると発表した。
人 1 / LLM 3.00 | 市は高校の授業料を無償化する。費用は年間約 60 億円である。
人 1 / LLM 2.29 | 県は来月から新しい図書館を開館する。
人 5 / LLM 3.00 | 市は給食費を無償化する。費用は約 6 億円で、寄付金などを充てる。


In [2]:
# 採点そのものを検証する: 人間の評点との相関
from scipy.stats import pearsonr, spearmanr

human = [h for _, h in items]
print(f"Spearman = {spearmanr(human, scores).statistic:.3f}  Pearson = {pearsonr(human, scores).statistic:.3f}")
print("良い要約 (人 4 以上) の平均:", round(np.mean([s for s, h in zip(scores, human) if h >= 4]), 2),
      "/ 悪い要約 (人 2 以下) の平均:", round(np.mean([s for s, h in zip(scores, human) if h <= 2]), 2))

Spearman = 0.860  Pearson = 0.360
良い要約 (人 4 以上) の平均: 3.0 / 悪い要約 (人 2 以下) の平均: 2.86


In [3]:
# 点数ではなく「はい / いいえ」で答えられる問いにして、「はい」の確率をスコアにする
YES_NO = [tokenizer(c, add_special_tokens=False)["input_ids"][0] for c in ["はい", "いいえ"]]

def p_faithful(source, summary):
    prompt = ("資料だけを根拠にして、次の「主張」が正しいと言えるかを判定してください。資料から正しいと分かるなら「はい」、"
              f"資料と食い違う、または資料からは分からない内容を含むなら「いいえ」と、一語で答えてください。\n\n資料: {source}\n主張: {summary}")
    return float(next_token_probs(prompt, YES_NO)[0])

yes_scores = [p_faithful(source, s) for s, _ in items]
for (s, h), y in zip(items, yes_scores):
    print(f"人 {h} / はいの確率 {y:.2f} | {s}")
print(f"Spearman = {spearmanr(human, yes_scores).statistic:.3f}")

人 5 / はいの確率 1.00 | 市は来年 4 月から小中学校の給食費を無償化する。費用は年間約 6 億円を見込む。
人 5 / はいの確率 1.00 | 市は給食費の無償化を発表した。市議会の一部からは財源を不安視する声がある。
人 5 / はいの確率 1.00 | 市は来年 4 月から給食費を無償化する。対象は約 1 万 2 千人である。
人 2 / はいの確率 0.00 | 市は給食費を無償化する。財源は国の補助金でまかなう。
人 3 / はいの確率 0.00 | 市は来年 4 月から給食費を無償化する。保護者の 9 割が賛成している。
人 1 / はいの確率 0.00 | 市は給食費の無償化を発表した。市議会は全会一致で賛成した。
人 1 / はいの確率 0.00 | 市は来年 4 月から給食費を値上げすると発表した。
人 1 / はいの確率 0.00 | 市は高校の授業料を無償化する。費用は年間約 60 億円である。
人 1 / はいの確率 0.00 | 県は来月から新しい図書館を開館する。
人 5 / はいの確率 1.00 | 市は給食費を無償化する。費用は約 6 億円で、寄付金などを充てる。
Spearman = 0.905


In [4]:
# 2 つの出力のどちらが良いかを選ばせる (一対比較)。提示順を入れ替えて 2 回聞き、順序による偏りを打ち消す
AB = [tokenizer(c, add_special_tokens=False)["input_ids"][-1] for c in ["A", "B"]]

def prefer_first(source, x, y):
    prompt = ("次の「元の文章」に対する 2 つの要約 A と B のうち、元の文章の内容をより正確に伝えているほうを選び、A か B の 1 文字だけで答えてください。\n\n"
              f"# 元の文章\n{source}\n\n# 要約 A\n{x}\n\n# 要約 B\n{y}")
    return float(next_token_probs(prompt, AB)[0])

def compare(source, x, y):
    forward, backward = prefer_first(source, x, y), 1 - prefer_first(source, y, x)
    return (forward + backward) / 2, forward, backward

pairs = [(0, 3), (1, 5), (2, 4), (9, 7), (0, 2)]      # (良いはずの要約, 比べる相手) の番号
for i, j in pairs:
    p, fwd, bwd = compare(source, items[i][0], items[j][0])
    print(f"前者が良い確率 {p:.2f} (前者を A に置く {fwd:.2f} / B に置く {bwd:.2f}) | 人の評点 {items[i][1]} vs {items[j][1]}")

前者が良い確率 1.00 (前者を A に置く 1.00 / B に置く 1.00) | 人の評点 5 vs 2
前者が良い確率 1.00 (前者を A に置く 0.99 / B に置く 1.00) | 人の評点 5 vs 1
前者が良い確率 1.00 (前者を A に置く 1.00 / B に置く 1.00) | 人の評点 5 vs 3
前者が良い確率 1.00 (前者を A に置く 1.00 / B に置く 1.00) | 人の評点 5 vs 1
前者が良い確率 0.71 (前者を A に置く 0.43 / B に置く 0.99) | 人の評点 5 vs 5


- 採点基準（ルーブリック）は、各点数が何を意味するかまで具体的に書く。観点（忠実性、流暢性、網羅性）ごとに別々に採点させる。
- 数字を生成させる代わりに、数字のトークンの確率から期待値をとると、同点が減り、細かい差が見える。
- ただし、この 3B のモデルは 1〜5 の採点ではほぼすべてに「3」を付けてしまう（10 件中 8 件が 3.00）。Spearman の相関は 0.86 と出るが、2.99 と 3.00 のようなわずかな差の順位を拾っているだけで、Pearson は 0.36、良い要約と悪い要約の平均も 3.0 と 2.86 でほとんど差がない。小さなモデルでは、段階評価より「はい / いいえ」で答えられる問いに分解するほうがはるかに安定する（同じ 10 件で、良い要約と悪い要約がはっきり分かれ、Spearman は 0.91）。
- LLM の評価には偏りがある: 提示順（先に出たほうを選びやすいなど）、長い出力を好む、自分と同系統のモデルの出力に甘い。一対比較では必ず順序を入れ替える。
- transformers 5 系では、このモデルのトークナイザーを読み込むときに分割の方式が学習時と違ってしまうので、コードで直している（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照）。直さないと採点の確率も変わる（ここでは「はい / いいえ」の Spearman が 0.91 から 0.87 に下がった。1〜5 の採点の Spearman は 0.86 から 0.94 に変わったが、前項のとおりわずかな差の順位を拾っているだけなので、良くなったわけではない）。
- 人手の評点との相関を確かめていない LLM 評価は、評価として使ってはいけない。ここの 10 件は動作確認用で、実際には 100 件以上で確かめる。
- 小さなモデルは、数値の誤りや細かい矛盾を見逃しやすい。重要な評価には、より大きなモデル（次項の API を使う方法）や、NLI・文字列照合などの機械的な検査を併用する。